# VQE Optimizer Comparison: SPSA vs CMA-ES vs Parameter-Shift

Compares three optimization methods on a ring-topology ZZ Hamiltonian:
- **Parameter-Shift** (exact gradient, 2P evaluations per step)
- **SPSA** (stochastic gradient, 2 evaluations per step)
- **CMA-ES** (population-based, derivative-free)

Tested on two ansatze:
- **QAOA** (problem + mixer layers)
- **HEA** (hardware-efficient ansatz: RY/RZ + CNOT ring)

All using **efficient contraction** (exact expectation values, no shot noise).

**Part 1**: Shallow circuits (N=6,8; L=1,2) — all 3 optimizers.

**Part 2**: Deep/complex circuits (N=6-14; L=2-4; rank=16) — CMA-ES vs ParamShift head-to-head.

## 1. Setup

In [ ]:
!pip install -q torch numpy pandas matplotlib seaborn hashable_list ordered_set
!pip install -q git+https://github.com/keunjunpark/TREV@real_form_autograd

In [ ]:
import math, time, gc
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.optimization.optimizer import Optimizer
from TREV.optimization.optimization import minimize
from TREV.optimization.gradients.batch_parameter_shift import BatchParameterShiftGradient
from TREV.optimization.gradients.spsa import SPSAGradient
from TREV.optimization.cma_es import CMAES, minimize_cma_es

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem_in_bytes / 1e9:.1f} GB')

## 2. Circuit and Hamiltonian Builders

In [ ]:
def ring_zz_hamiltonian(n_qubits):
    """ZZ ring Hamiltonian: sum_i Z_i Z_{(i+1) % N}."""
    terms, coeffs = [], []
    for i in range(n_qubits):
        j = (i + 1) % n_qubits
        pauli = ['I'] * n_qubits
        pauli[i] = 'Z'
        pauli[j] = 'Z'
        terms.append(''.join(pauli))
        coeffs.append(1.0)
    return Hamiltonian(n_qubits, terms, coeffs)


def build_qaoa_circuit(n_qubits, p_layers, rank):
    """QAOA ansatz on ring topology.
    
    Problem layer: CNOT-RZ-CNOT per ring edge (ZZ interaction).
    Mixer layer: RX on each qubit.
    Params per layer: N (RZ edges) + N (RX mixer) = 2N.
    Total params: 2N * p_layers.
    """
    c = Circuit(num_qubit=n_qubits, rank=rank, device=DEVICE)
    for i in range(n_qubits):
        c.h(i)
    for _ in range(p_layers):
        # Problem layer: ZZ via CNOT-RZ-CNOT on ring
        for i in range(n_qubits):
            j = (i + 1) % n_qubits
            c.cx(i, j)
            c.rz(j)
            c.cx(i, j)
        # Mixer layer
        for i in range(n_qubits):
            c.rx(i)
    return c


def build_hea_circuit(n_qubits, layers, rank):
    """Hardware-efficient ansatz on ring topology.
    
    Each layer: CNOT ring + RY + RZ per qubit.
    Params per layer: 2N (RY + RZ).
    Total params: 2N * layers.
    """
    c = Circuit(num_qubit=n_qubits, rank=rank, device=DEVICE)
    for i in range(n_qubits):
        c.h(i)
    for _ in range(layers):
        for i in range(n_qubits):
            c.cx(i, (i + 1) % n_qubits)
        for i in range(n_qubits):
            c.ry(i)
            c.rz(i)
    return c


# Quick test
for name, builder in [('QAOA', build_qaoa_circuit), ('HEA', build_hea_circuit)]:
    c = builder(6, 2, 8)
    print(f'{name}: {c.num_qubit} qubits, {c.params_size} params, {len(c.gates)} gates')

---
# Part 1: Shallow Circuits (All 3 Optimizers)

## 3. Experiment Configuration

## 4. Run All Experiments

## 5. Build Results DataFrame

## 6. Convergence Plots

## 7. Convergence vs Wall-Clock Time

## 8. Convergence vs Total Circuit Evaluations

## 9. Per-Iteration Timing

## 10. Running-Best Expectation Value

## 11. Summary Table

## 12. Speedup Analysis

---
# Part 2: Deep Circuits — CMA-ES vs Parameter-Shift

Stress-test with larger qubit counts, deeper layers, and higher bond dimension.
Does CMA-ES's O(n^2) covariance matrix start hurting at P=72, or does it still beat ParamShift's 2P evaluations per step?

## 13. Deep Circuit Configuration

## 14. Run Deep Experiments

## 15. Deep Results DataFrame

## 16. Deep Convergence by Iteration

## 17. Deep Convergence vs Wall-Clock Time

## 18. Deep Convergence vs Circuit Evaluations

## 19. Gap to Ground State

## 20. Wall-Clock Speedup

## 21. Deep Summary Table

---
# Part 3: TSP QAOA — CMA-ES vs Parameter-Shift

Real-world VQE benchmark: Travelling Salesman Problem encoded as QAOA on a ring-topology tensor network.

- TSP instances with **n_cities=3** (9 qubits) and **n_cities=4** (16 qubits)
- QAOA **reps=1,2** depth
- **3 random seeds** per configuration
- Hamiltonian from QUBO Ising encoding (hundreds of Pauli terms)
- QAOA-parameterized optimization (2*reps params projected via Jacobian)

## 22. TSP Dependencies

In [ ]:
!pip install -q qiskit qiskit-optimization

import itertools
from qiskit.circuit.library import QAOAAnsatz
from qiskit import transpile as qk_transpile
from qiskit.transpiler import CouplingMap
from qiskit_optimization.applications import Tsp
from qiskit_optimization.converters import QuadraticProgramToQubo
from TREV.transpile import from_qiskit
from TREV.measure.right_suffix_sampling import argmax_bitstring_tr_right_suffix

TSP_BASIS_GATES = ['swap', 'rzz', 'rx', 'h']
print('TSP dependencies loaded.')

## 23. TSP Helpers

In [ ]:
def make_tsp_ising(n_cities, seed):
    """Generate TSP instance and convert to Ising Hamiltonian."""
    tsp_inst = Tsp.create_random_instance(n_cities, seed=seed)
    qp = tsp_inst.to_quadratic_program()
    qubo = QuadraticProgramToQubo().convert(qp)
    qubitOp, offset = qubo.to_ising()
    return qubitOp, offset, tsp_inst


def normalize_ising(qubitOp):
    max_coeff = max(abs(float(c.real)) for c in qubitOp.coeffs)
    if max_coeff > 0:
        qubitOp = qubitOp / max_coeff
    return qubitOp, max_coeff


def build_trev_hamiltonian(qubitOp):
    pauli_strings, coefficients = [], []
    for elm in qubitOp:
        pauli_strings.append(str(elm.paulis[0][::-1]))
        coefficients.append(float(elm.coeffs[0].real))
    return Hamiltonian(len(pauli_strings[0]), pauli_strings, coefficients)


def get_distance_matrix(tsp_inst):
    G = tsp_inst.graph
    n = len(G.nodes)
    dist = np.zeros((n, n))
    for i, j, data in G.edges(data=True):
        w = data.get('weight', 1.0)
        dist[i][j] = w
        dist[j][i] = w
    return dist


def solve_tsp_brute(dist):
    n = dist.shape[0]
    best_cost, best_tour = float('inf'), None
    for perm in itertools.permutations(range(n)):
        cost = sum(dist[perm[i], perm[(i + 1) % n]] for i in range(n))
        if cost < best_cost:
            best_cost, best_tour = cost, list(perm)
    return best_cost, best_tour


def decode_tsp_bitstring(bits, n_cities, qubit_perm=None):
    if isinstance(bits, str):
        bits = [int(b) for b in bits]
    bits = list(bits)
    if qubit_perm is not None:
        inv_perm = [0] * len(qubit_perm)
        for i, p in enumerate(qubit_perm):
            inv_perm[p] = i
        bits = [bits[inv_perm[i]] for i in range(len(bits))]
    n = n_cities
    N = n * n
    if len(bits) < N:
        return False, None
    matrix = np.array(bits[:N]).reshape(n, n)
    if not (np.all(matrix.sum(axis=1) == 1) and np.all(matrix.sum(axis=0) == 1)):
        return False, None
    tour = [int(np.argmax(matrix[:, t])) for t in range(n)]
    return True, tour


def compute_tour_cost(tour, dist):
    n = len(tour)
    return sum(dist[tour[i]][tour[(i+1) % n]] for i in range(n))


def build_qaoa_mapping(routed_qc, rank, device, fuse_zz_swap=True):
    """Build linear mapping from QAOA params to TREV theta.
    Returns: (trev_circuit, theta_base, J_matrix, qaoa_param_names)
    """
    params = routed_qc.parameters
    sorted_params = sorted(params, key=lambda p: p.name)
    qaoa_param_names = [p.name for p in sorted_params]
    K = len(qaoa_param_names)

    zero_bind = {p: 0.0 for p in params}
    qc_zero = routed_qc.assign_parameters(zero_bind)
    trev_circuit, theta_base = from_qiskit(qc_zero, fuse_zz_swap=fuse_zz_swap,
                                            rank=rank, device=device)
    P = theta_base.shape[0]

    J = torch.zeros(P, K, dtype=theta_base.dtype)
    for i, param in enumerate(sorted_params):
        unit_bind = {p: 0.0 for p in params}
        unit_bind[param] = 1.0
        qc_unit = routed_qc.assign_parameters(unit_bind)
        _, theta_unit = from_qiskit(qc_unit, fuse_zz_swap=fuse_zz_swap,
                                     rank=rank, device=device)
        J[:, i] = (theta_unit - theta_base).cpu()

    return trev_circuit, theta_base.cpu(), J, qaoa_param_names


print('TSP helpers loaded.')

## 24. TSP QAOA Optimization (ParamShift & CMA-ES)

In [ ]:
def run_tsp_qaoa_paramshift(circuit, theta_base, J, hamil, qaoa_x0,
                            n_iters, lr, shots=0,
                            measure_method=MeasureMethod.EFFICIENT_CONTRACTION):
    """QAOA-param optimization with parameter-shift gradient."""
    device = circuit.device
    theta_base_dev = theta_base.to(device)
    J_dev = J.to(device)
    qaoa_params = qaoa_x0.clone().detach().to(device)

    grad_estimator = BatchParameterShiftGradient(
        shift=math.pi / 2, batch_size=None, shots=shots,
        measure_method=measure_method, depth=1)

    qaoa_params.requires_grad_(True)
    adam = torch.optim.Adam([qaoa_params], lr=lr)

    exp_values, best_results, iter_times = [], [], []
    start = time.time()

    with torch.no_grad():
        for epoch in range(n_iters):
            it_time = time.time()
            adam.zero_grad()
            full_theta = theta_base_dev + J_dev @ qaoa_params
            full_grad = grad_estimator.run(full_theta.detach(), circuit, hamil)
            qaoa_params.grad = J_dev.T @ full_grad
            adam.step()
            if device == 'cuda': torch.cuda.synchronize()
            iter_times.append(time.time() - it_time)

            full_theta_eval = theta_base_dev + J_dev @ qaoa_params
            ev = circuit.get_expectation_value(full_theta_eval, hamil, measure_method)
            exp_values.append(float(ev))

            tensor = circuit.build_tensor(full_theta_eval)
            best_results.append(argmax_bitstring_tr_right_suffix(tensor))

            if epoch % 10 == 0:
                gc.collect()
                if device == 'cuda': torch.cuda.empty_cache()

    if hasattr(grad_estimator, '_gpu_pool') and grad_estimator._gpu_pool is not None:
        grad_estimator._gpu_pool.shutdown()
    total_time = time.time() - start
    return exp_values, best_results, iter_times, total_time


def run_tsp_qaoa_cmaes(circuit, theta_base, J, hamil, qaoa_x0,
                       n_gens, sigma=0.5,
                       measure_method=MeasureMethod.EFFICIENT_CONTRACTION):
    """QAOA-param optimization with CMA-ES in QAOA subspace."""
    device = circuit.device
    theta_base_dev = theta_base.to(device)
    J_dev = J.to(device)
    K = J.shape[1]

    # CMA-ES operates in K-dimensional QAOA space
    cma = CMAES(sigma=sigma, measure_method=measure_method, shots=0)
    s = cma._init_state(K, device)
    s['mean'] = qaoa_x0.clone().double().to(device)

    exp_values, best_results, iter_times = [], [], []
    start = time.time()
    last_bitstring = None

    def evaluate_qaoa_pop(qaoa_pop):
        """Map QAOA population to full theta, evaluate all at once."""
        # qaoa_pop: (lam, K)
        full_pop = theta_base_dev + qaoa_pop.to(device) @ J_dev.T  # (lam, P)
        return circuit.get_expectation_value(
            full_pop, hamil, measure_method, 0)

    for gen in range(n_gens):
        it_time = time.time()
        best_cost, best_qaoa = cma._step(s, evaluate_qaoa_pop)
        if device == 'cuda': torch.cuda.synchronize()
        iter_times.append(time.time() - it_time)
        exp_values.append(best_cost)

        # Best bitstring every 10 gens
        if gen % 10 == 0 or gen == n_gens - 1:
            with torch.no_grad():
                full_theta = theta_base_dev + J_dev @ best_qaoa.float()
                tensor = circuit.build_tensor(full_theta)
                last_bitstring = argmax_bitstring_tr_right_suffix(tensor)
        best_results.append(last_bitstring)

        if gen % 10 == 0:
            gc.collect()
            if device == 'cuda': torch.cuda.empty_cache()

    total_time = time.time() - start
    return exp_values, best_results, iter_times, total_time


print('TSP optimization functions loaded.')

## 25. TSP Experiment Configuration

In [ ]:
# ── TSP Grid ──
TSP_NC_LIST   = [3, 4]           # 3 cities = 9 qubits, 4 = 16 qubits
TSP_REPS_LIST = [1, 2]           # QAOA depth
TSP_SEEDS     = [0, 1, 2]        # 3 random seeds
TSP_RANK      = 16               # bond dimension
TSP_ITERS     = 200              # iterations/generations
TSP_LR        = 0.05             # Adam learning rate for param-shift
TSP_CMA_SIGMA = 0.3              # CMA-ES initial step size
TSP_MEASURE   = MeasureMethod.EFFICIENT_CONTRACTION

tsp_configs = []
for nc in TSP_NC_LIST:
    for reps in TSP_REPS_LIST:
        for seed in TSP_SEEDS:
            tsp_configs.append((nc, reps, seed))

print(f'{len(tsp_configs)} TSP configs x 2 optimizers = {len(tsp_configs) * 2} runs')

## 26. Run TSP QAOA Experiments

In [ ]:
tsp_results = []
tsp_ground_truth = {}  # (nc, seed) -> (opt_cost, opt_tour, dist)

for idx, (nc, reps, seed) in enumerate(tsp_configs):
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Build TSP problem
    qubitOp, offset, tsp_inst = make_tsp_ising(nc, seed)
    qubitOp_n, scale = normalize_ising(qubitOp)
    N = qubitOp_n.num_qubits
    hamil = build_trev_hamiltonian(qubitOp_n)

    if (nc, seed) not in tsp_ground_truth:
        dist = get_distance_matrix(tsp_inst)
        opt_cost, opt_tour = solve_tsp_brute(dist)
        tsp_ground_truth[(nc, seed)] = (opt_cost, opt_tour, dist)
        print(f'TSP(nc={nc}, seed={seed}): opt_cost={opt_cost:.2f}, N={N} qubits')

    # Build QAOA circuit and route to ring
    qaoa = QAOAAnsatz(qubitOp_n, reps=reps)
    optimized = qk_transpile(qaoa, optimization_level=3, basis_gates=TSP_BASIS_GATES)
    cm = CouplingMap.from_ring(N)
    routed = qk_transpile(optimized, coupling_map=cm, optimization_level=1,
                          basis_gates=TSP_BASIS_GATES, seed_transpiler=seed)

    # Build QAOA mapping
    trev_circ, theta_base, J, pnames = build_qaoa_mapping(
        routed, TSP_RANK, DEVICE, fuse_zz_swap=True)
    K = len(pnames)
    P = theta_base.shape[0]

    # Initial QAOA params
    gen = torch.Generator().manual_seed(seed * 10000 + nc * 100 + reps)
    qaoa_x0 = 0.1 * torch.randn(K, generator=gen)

    print(f'\n{"="*60}')
    print(f'[{idx+1}/{len(tsp_configs)}] TSP nc={nc} reps={reps} seed={seed} | '
          f'N={N} qubits, K={K} QAOA params, P={P} TREV params')
    print(f'{"="*60}')

    # ── ParamShift ──
    print(f'  ParamShift ...')
    ps_ev, ps_br, ps_times, ps_total = run_tsp_qaoa_paramshift(
        trev_circ, theta_base, J, hamil, qaoa_x0,
        n_iters=TSP_ITERS, lr=TSP_LR, measure_method=TSP_MEASURE)
    tsp_results.append(dict(
        optimizer='ParamShift', n_cities=nc, reps=reps, seed=seed,
        n_qubits=N, K=K, P=P, rank=TSP_RANK,
        exp_values=ps_ev, best_results=ps_br,
        iter_times=ps_times, total_time=ps_total,
        qubit_perm=trev_circ.qubit_perm))
    print(f'    min={min(ps_ev):.4f}  final={ps_ev[-1]:.4f}  time={ps_total:.1f}s')
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

    # ── CMA-ES ──
    print(f'  CMA-ES ...')
    cma_ev, cma_br, cma_times, cma_total = run_tsp_qaoa_cmaes(
        trev_circ, theta_base, J, hamil, qaoa_x0,
        n_gens=TSP_ITERS, sigma=TSP_CMA_SIGMA, measure_method=TSP_MEASURE)
    tsp_results.append(dict(
        optimizer='CMA-ES', n_cities=nc, reps=reps, seed=seed,
        n_qubits=N, K=K, P=P, rank=TSP_RANK,
        exp_values=cma_ev, best_results=cma_br,
        iter_times=cma_times, total_time=cma_total,
        qubit_perm=trev_circ.qubit_perm))
    print(f'    min={min(cma_ev):.4f}  final={cma_ev[-1]:.4f}  time={cma_total:.1f}s')

    del trev_circ, theta_base, J
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print(f'\nDone! {len(tsp_results)} TSP runs.')

## 27. TSP Results

In [ ]:
tsp_records = []
for r in tsp_results:
    ev = r['exp_values']
    nc, seed = r['n_cities'], r['seed']
    opt_cost, opt_tour, dist = tsp_ground_truth[(nc, seed)]
    qperm = r.get('qubit_perm')

    # Track best feasible tour
    best_ratio = 0.0
    n_feasible = 0
    for bits in r['best_results']:
        feasible, tour = decode_tsp_bitstring(bits, nc, qperm)
        if feasible and tour is not None:
            n_feasible += 1
            cost = compute_tour_cost(tour, dist)
            ratio = opt_cost / cost if cost > 0 else 0.0
            best_ratio = max(best_ratio, ratio)

    K = r['K']
    epi = 2 * K if r['optimizer'] == 'ParamShift' else (4 + int(3 * math.log(K)))
    tsp_records.append({
        'optimizer': r['optimizer'],
        'n_cities': nc, 'n_qubits': r['n_qubits'],
        'reps': r['reps'], 'seed': seed,
        'K': K, 'P': r['P'], 'rank': r['rank'],
        'min_E': min(ev), 'final_E': ev[-1],
        'best_accuracy': best_ratio,
        'n_feasible': n_feasible,
        'evals_per_iter': epi,
        'med_iter_ms': np.median(r['iter_times'][2:]) * 1000,
        'total_time_s': r['total_time'],
    })

tsp_df = pd.DataFrame(tsp_records)
display(tsp_df.round(3))

## 28. TSP Convergence Plots

In [ ]:
TSP_COLORS = {'ParamShift': 'tab:blue', 'CMA-ES': 'tab:green'}

tsp_unique = list(dict.fromkeys(
    [(r['n_cities'], r['reps']) for r in tsp_results]))
n_cols_t = len(tsp_unique)

# --- By Iteration (one line per seed, averaged would hide variance) ---
fig, axes = plt.subplots(1, n_cols_t, figsize=(6*n_cols_t, 5), squeeze=False)
fig.suptitle('TSP QAOA: Convergence by Iteration (all seeds)', fontsize=14, y=1.02)
for i, (nc, reps) in enumerate(tsp_unique):
    ax = axes[0][i]
    for r in tsp_results:
        if r['n_cities'] == nc and r['reps'] == reps:
            color = TSP_COLORS[r['optimizer']]
            label = r['optimizer'] if r['seed'] == TSP_SEEDS[0] else None
            ax.plot(r['exp_values'], color=color, alpha=0.5, linewidth=1,
                    label=label)
    # Plot seed-averaged
    for opt in ['ParamShift', 'CMA-ES']:
        evs = [r['exp_values'] for r in tsp_results
               if r['n_cities']==nc and r['reps']==reps and r['optimizer']==opt]
        if evs:
            min_len = min(len(e) for e in evs)
            avg = np.mean([e[:min_len] for e in evs], axis=0)
            ax.plot(avg, color=TSP_COLORS[opt], linewidth=2.5,
                    label=f'{opt} (avg)', linestyle='--')
    N = nc * nc
    ax.set_title(f'nc={nc} (N={N}) reps={reps}', fontsize=11)
    ax.set_xlabel('Iteration')
    if i == 0: ax.set_ylabel('$\\langle H \\rangle$')
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

# --- By Wall-Clock Time ---
fig, axes = plt.subplots(1, n_cols_t, figsize=(6*n_cols_t, 5), squeeze=False)
fig.suptitle('TSP QAOA: Convergence vs Wall-Clock Time', fontsize=14, y=1.02)
for i, (nc, reps) in enumerate(tsp_unique):
    ax = axes[0][i]
    for r in tsp_results:
        if r['n_cities'] == nc and r['reps'] == reps:
            cum_t = np.cumsum(r['iter_times'])
            color = TSP_COLORS[r['optimizer']]
            label = r['optimizer'] if r['seed'] == TSP_SEEDS[0] else None
            ax.plot(cum_t, r['exp_values'], color=color, alpha=0.5,
                    linewidth=1, label=label)
    ax.set_title(f'nc={nc} reps={reps}', fontsize=11)
    ax.set_xlabel('Time (s)')
    if i == 0: ax.set_ylabel('$\\langle H \\rangle$')
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

## 29. TSP Solution Quality

In [ ]:
# Accuracy and feasibility comparison
print('=== TSP Solution Quality ===')
print(f'{"Config":<30} {"Optimizer":<12} {"Accuracy":>10} {"Feasible":>10} {"min_E":>10} {"Time":>10}')
print('-' * 85)

for nc, reps in tsp_unique:
    for opt in ['ParamShift', 'CMA-ES']:
        sub = tsp_df[(tsp_df['n_cities']==nc) & (tsp_df['reps']==reps) & (tsp_df['optimizer']==opt)]
        avg_acc = sub['best_accuracy'].mean()
        avg_feas = sub['n_feasible'].mean()
        avg_E = sub['min_E'].mean()
        avg_time = sub['total_time_s'].mean()
        N = nc * nc
        print(f'nc={nc} (N={N:>2}) reps={reps}  seeds avg  '
              f'{opt:<12} {avg_acc:>10.3f} {avg_feas:>10.1f} {avg_E:>10.4f} {avg_time:>10.1f}s')
    print()

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy
ax = axes[0]
tsp_df['label'] = tsp_df.apply(lambda r: f"nc={r['n_cities']} r={r['reps']}", axis=1)
pivot_acc = tsp_df.pivot_table(index='label', columns='optimizer',
                               values='best_accuracy', aggfunc='mean')
pivot_acc[['ParamShift', 'CMA-ES']].plot.bar(
    ax=ax, color=[TSP_COLORS['ParamShift'], TSP_COLORS['CMA-ES']],
    edgecolor='black', width=0.7)
ax.set_title('TSP: Best Accuracy (avg over seeds)', fontsize=12)
ax.set_ylabel('Optimality Ratio')
ax.set_ylim(0, 1.1)
ax.axhline(1.0, color='red', ls=':', alpha=0.4)
ax.tick_params(axis='x', rotation=0)

# Timing
ax = axes[1]
pivot_time = tsp_df.pivot_table(index='label', columns='optimizer',
                                values='total_time_s', aggfunc='mean')
pivot_time[['ParamShift', 'CMA-ES']].plot.bar(
    ax=ax, color=[TSP_COLORS['ParamShift'], TSP_COLORS['CMA-ES']],
    edgecolor='black', width=0.7)
ax.set_title('TSP: Total Wall-Clock Time (avg over seeds)', fontsize=12)
ax.set_ylabel('Time (s)')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 30. TSP Summary Table

In [ ]:
print('=== TSP QAOA: CMA-ES vs ParamShift ===')
print(f'Rank={TSP_RANK}, Iters={TSP_ITERS}, Seeds={TSP_SEEDS}')
print()

tsp_summary = tsp_df[['optimizer', 'n_cities', 'n_qubits', 'reps', 'seed',
                      'K', 'P', 'min_E', 'best_accuracy', 'n_feasible',
                      'evals_per_iter', 'med_iter_ms', 'total_time_s']].copy().round(3)

try:
    display(tsp_summary.style.background_gradient(
        subset=['best_accuracy'], cmap='RdYlGn', vmin=0, vmax=1
    ).background_gradient(
        subset=['total_time_s'], cmap='RdYlGn_r'
    ))
except:
    print(tsp_summary.to_string(index=False))